# NOTEBOOK 16 · Seq2seq con atención de Bahdanau

## Del contexto fijo a un contexto distinto en cada paso

Partimos de `nb05_bis.ipynb`: reutilizamos tokenización, vocabularios, lotes, pérdida, entrenamiento y evaluación. Conservamos la LSTM **unidireccional** y sus estados iniciales para aislar mejor el efecto de añadir atención.

Antes, el decoder solo recibía los estados finales del encoder. Ahora conserva también las salidas de todas sus posiciones y aprende a combinarlas en cada paso.

Implementamos la **atención aditiva de Bahdanau** adaptada a nuestras LSTM. No es una reproducción completa de la arquitectura original, que utiliza otros componentes. Referencia: [Bahdanau, Cho y Bengio, Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473).

Si existe `artifacts/nb05_bis/experiment.json`, reutilizamos sus particiones, vocabularios e hiperparámetros. Si no existe, este notebook funciona de forma independiente con los mismos valores predeterminados. Para comparar resultados entrenados, ejecuta y guarda primero el notebook sin atención.

**Requisitos:** Python, PyTorch y Matplotlib. SacreBLEU es opcional. Ejecuta las celdas en orden desde el directorio del notebook.

In [ ]:
from pathlib import Path
from collections import Counter
from itertools import zip_longest
import math
import random
import re
import time

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# CPU también funciona. Las GPU NVIDIA se usan si están disponibles.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)
import json

BASELINE_FILE = Path("artifacts/nb05_bis/experiment.json")
baseline = None
if BASELINE_FILE.exists():
    baseline = json.loads(BASELINE_FILE.read_text(encoding="utf-8"))
    SEED = baseline["config"]["seed"]
    random.seed(SEED)
    torch.manual_seed(SEED)
    print("Se reutilizará el experimento sin atención:", BASELINE_FILE)
else:
    print("No hay experimento guardado: se usarán las particiones predeterminadas.")

## 1. Datos y particiones

Leemos pares alineados de Europarl. Para una primera práctica limitamos la longitud y el número de ejemplos. El subconjunto son los primeros pares válidos del fichero; no es una muestra representativa de todo Europarl. Se pueden ampliar estos límites después.

Eliminamos pares duplicados y asignamos cada frase inglesa a una única partición. Así, una misma entrada no aparece en entrenamiento y prueba, aunque tenga varias traducciones. Esto no elimina paráfrasis ni repeticiones entre documentos; una evaluación más exigente separaría debates completos.

La tokenización separa palabras y signos de puntuación. Es deliberadamente sencilla; no utiliza subpalabras. Las palabras que no estén en el vocabulario se representarán mediante `<unk>`.

In [ ]:
SOURCE_FILE = Path("data/europarl/europarl-v7.es-en.en")
TARGET_FILE = Path("data/europarl/europarl-v7.es-en.es")
MAX_WORDS = 30             # Sin contar los tokens especiales.
MAX_PAIRS = 30_000         # Usa None para conservar todos los pares válidos.
MIN_FREQUENCY = 2
MAX_VOCAB_SIZE = 12_000    # Incluye los cuatro tokens especiales.


def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)


def read_pairs(source_file, target_file, max_words, max_pairs):
    pairs = []
    seen = set()
    with source_file.open(encoding="utf-8") as source, target_file.open(encoding="utf-8") as target:
        for line_number, (en, es) in enumerate(zip_longest(source, target), start=1):
            if en is None or es is None:
                raise ValueError(f"Ficheros desalineados en la línea {line_number}")
            en_tokens, es_tokens = tokenize(en), tokenize(es)
            if not (1 <= len(en_tokens) <= max_words and 1 <= len(es_tokens) <= max_words):
                continue
            pair = (tuple(en_tokens), tuple(es_tokens))
            if pair in seen:
                continue
            seen.add(pair)
            pairs.append(pair)
            if max_pairs is not None and len(pairs) >= max_pairs:
                break
    return pairs


def split_pairs(pairs, seed):
    sources = sorted({en for en, es in pairs})
    if len(sources) < 20:
        raise ValueError("Se necesitan al menos 20 entradas distintas para esta práctica.")
    random.Random(seed).shuffle(sources)
    n_valid = max(1, int(0.1 * len(sources)))
    n_test = max(1, int(0.1 * len(sources)))
    valid_sources = set(sources[:n_valid])
    test_sources = set(sources[n_valid:n_valid + n_test])
    train, valid, test = [], [], []
    for pair in pairs:
        en, es = pair
        if en in valid_sources:
            valid.append(pair)
        elif en in test_sources:
            test.append(pair)
        else:
            train.append(pair)
    return train, valid, test


if baseline is not None:
    previous = baseline["config"]
    SOURCE_FILE = Path(previous["source_file"])
    TARGET_FILE = Path(previous["target_file"])
    MAX_WORDS, MAX_PAIRS = previous["max_words"], previous["max_pairs"]
    MIN_FREQUENCY = previous["min_frequency"]
    MAX_VOCAB_SIZE = previous["max_vocab_size"]
    # Recuperamos los tokens guardados; no releemos ni repartimos el corpus.
    train_pairs = baseline["train_pairs"]
    valid_pairs = baseline["valid_pairs"]
    test_pairs = baseline["test_pairs"]
else:
    pairs = read_pairs(SOURCE_FILE, TARGET_FILE, MAX_WORDS, MAX_PAIRS)
    train_pairs, valid_pairs, test_pairs = split_pairs(pairs, SEED)
print("Pares de entrenamiento, validación y prueba:",
      len(train_pairs), len(valid_pairs), len(test_pairs))
print("Ejemplo:", train_pairs[0])

## 2. Vocabularios y lotes

Los identificadores especiales son iguales en ambos idiomas, pero cada idioma tiene su propio vocabulario y su propia capa de embeddings.

- `<pad>` rellena las secuencias de un lote.
- `<sos>` inicia la entrada del decoder.
- `<eos>` marca el final que debe aprender a predecir.
- `<unk>` representa palabras desconocidas.

No añadimos un `<pad>` extra a cada frase. El relleno se añade únicamente al formar el lote.

In [ ]:
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]


class Vocabulary:
    def __init__(self, sentences, min_frequency, max_size):
        counts = Counter(token for sentence in sentences for token in sentence)
        words = [word for word, count in counts.most_common()
                 if count >= min_frequency and word not in SPECIAL_TOKENS]
        self.itos = SPECIAL_TOKENS + words[:max_size - len(SPECIAL_TOKENS)]
        self.stoi = {word: index for index, word in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(token, UNK) for token in tokens]

    def decode(self, ids):
        words = []
        for index in ids:
            if index == EOS:
                break
            if index not in (PAD, SOS):
                words.append(self.itos[index])
        return words


src_vocab = Vocabulary((en for en, es in train_pairs), MIN_FREQUENCY, MAX_VOCAB_SIZE)
tgt_vocab = Vocabulary((es for en, es in train_pairs), MIN_FREQUENCY, MAX_VOCAB_SIZE)


if baseline is not None:
    # Reutilizamos incluso los identificadores exactos del experimento anterior.
    for vocab, key in [(src_vocab, "src_tokens"), (tgt_vocab, "tgt_tokens")]:
        vocab.itos = baseline[key]
        vocab.stoi = {word: index for index, word in enumerate(vocab.itos)}


class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.examples = []
        for en, es in pairs:
            source = torch.tensor(src_vocab.encode(en) + [EOS], dtype=torch.long)
            target = torch.tensor([SOS] + tgt_vocab.encode(es) + [EOS], dtype=torch.long)
            self.examples.append((source, target))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_batch(examples):
    sources, targets = zip(*examples)
    source_lengths = torch.tensor([len(source) for source in sources])
    sources = pad_sequence(sources, batch_first=True, padding_value=PAD)
    targets = pad_sequence(targets, batch_first=True, padding_value=PAD)
    return sources, source_lengths, targets


BATCH_SIZE = baseline["config"]["batch_size"] if baseline is not None else 32
train_dataset = TranslationDataset(train_pairs, src_vocab, tgt_vocab)
valid_dataset = TranslationDataset(valid_pairs, src_vocab, tgt_vocab)
test_dataset = TranslationDataset(test_pairs, src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_batch,
                          generator=torch.Generator().manual_seed(SEED))
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_batch)

print("Vocabularios inglés/español:", len(src_vocab), len(tgt_vocab))
sources, source_lengths, targets = next(iter(train_loader))
print("Entrada:", sources.shape, "Longitudes:", source_lengths.shape)
print("Objetivo con <sos> y <eos>:", targets.shape)

def unknown_rate(pairs, vocab, language_index):
    tokens = [token for pair in pairs for token in pair[language_index]]
    return sum(token not in vocab.stoi for token in tokens) / len(tokens)

print(f"Palabras desconocidas en prueba (EN): {unknown_rate(test_pairs, src_vocab, 0):.1%}")
print(f"Palabras desconocidas en prueba (ES): {unknown_rate(test_pairs, tgt_vocab, 1):.1%}")

## 3. Encoder: conservar también las salidas temporales

La única diferencia respecto al encoder sin atención es que recuperamos `outputs`, además de `(hidden, cell)`.

La LSTM recibe secuencias empaquetadas para ignorar el relleno. `pad_packed_sequence` reconstruye un tensor de forma `(lote, longitud_entrada, dimensión_oculta)`, en el orden original. Cada posición contiene la salida de la última capa LSTM para ese token.

Los estados finales siguen inicializando el decoder. Las salidas temporales serán la memoria que consultará la atención. Como conservamos un encoder unidireccional, cada salida resume el prefijo leído hasta esa posición.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                            num_layers=num_layers, batch_first=True)

    def forward(self, source, source_lengths):
        embedded = self.embedding(source)
        packed = pack_padded_sequence(embedded, source_lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        packed_outputs, (hidden, cell) = self.lstm(packed)
        outputs, _ = pad_packed_sequence(
            packed_outputs, batch_first=True, total_length=source.size(1))
        return outputs, hidden, cell

## 4. Atención aditiva y decoder

En el paso $t$ calculamos una puntuación para cada posición $i$ de la entrada:

$$e_{t,i}=v^\top\tanh(W_q s_{t-1}+W_k h_i).$$

- $s_{t-1}$: estado oculto de la última capa del decoder **antes** de procesar el token actual (consulta).
- $h_i$: salida del encoder en la posición $i$.
- $W_q$, $W_k$ y $v$: transformaciones aprendidas durante el entrenamiento.

Ponemos las puntuaciones del relleno a menos infinito y aplicamos softmax sobre las posiciones de entrada:

$$\alpha_{t,i}=\operatorname{softmax}_i(e_{t,i}),\qquad c_t=\sum_i\alpha_{t,i}h_i.$$

Los pesos válidos suman uno. La suma ponderada es diferenciable: la pérdida entrena tanto el encoder como el mecanismo de atención. No seleccionamos una posición con `argmax`.

Concatenamos el contexto con el embedding del token para alimentar la LSTM. Para predecir, concatenamos también su salida con el contexto. El decoder devuelve los pesos para poder visualizarlos.

### Formas de los tensores

| Tensor | Forma |
|---|---|
| Consulta | `(lote, hidden_dim)` |
| Salidas del encoder | `(lote, longitud_entrada, hidden_dim)` |
| Máscara válida | `(lote, longitud_entrada)` |
| Pesos | `(lote, longitud_entrada)` |
| Contexto | `(lote, hidden_dim)` |

Conservamos el tamaño oculto del modelo anterior, pero el modelo con atención tiene más parámetros. La comparación evalúa estas dos arquitecturas; no iguala exactamente su número de parámetros.

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.query_layer = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key_layer = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.score_layer = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, query, encoder_outputs, source_mask):
        # query: (lote, H) -> (lote, 1, H), para combinarla con cada posición.
        query_features = self.query_layer(query).unsqueeze(1)
        key_features = self.key_layer(encoder_outputs)
        scores = self.score_layer(torch.tanh(query_features + key_features)).squeeze(-1)

        # True significa token real. Nunca atenderemos a las posiciones de relleno.
        scores = scores.masked_fill(~source_mask, -torch.inf)
        weights = torch.softmax(scores, dim=-1)

        # (lote, 1, longitud) @ (lote, longitud, H) -> (lote, 1, H)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, weights


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.attention = BahdanauAttention(hidden_dim)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim,
                            num_layers=num_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim + hidden_dim, vocab_size)

    def forward(self, token_ids, hidden, cell, encoder_outputs, source_mask):
        embedded = self.embedding(token_ids)
        # [-1] selecciona la última capa, no el último instante temporal.
        context, weights = self.attention(hidden[-1], encoder_outputs, source_mask)
        lstm_input = torch.cat((embedded, context), dim=-1).unsqueeze(1)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        features = torch.cat((output[:, 0, :], context), dim=-1)
        logits = self.output_layer(features)
        return logits, hidden, cell, weights

## 5. Seq2seq y teacher forcing

Para el objetivo `<sos> mi perro <eos>`, el primer paso recibe `<sos>` y debe predecir `mi`. Después recibe `mi` (teacher forcing) o su propia predicción. Nunca recibe la palabra que está intentando predecir en ese mismo paso.

La decisión de teacher forcing se toma por paso para todo el lote. En la rama sin teacher forcing realimentamos el **identificador predicho**, que pasa por la misma capa de embeddings utilizada con los tokens reales. `argmax` no es diferenciable; la pérdida de cada paso sí entrena sus logits y los estados recurrentes.

Durante el entrenamiento recorremos la longitud del objetivo del lote; los `<pad>` no contribuyen a la pérdida. La generación libre tendrá un método separado y no recibirá objetivos.

**Cambio respecto al notebook anterior:** el encoder devuelve su memoria completa y creamos `source_mask = source != PAD`. En cada paso pasamos ambos al decoder. El resto del teacher forcing se conserva.

`generate` mantiene su salida habitual para reutilizar la evaluación. Con `return_attention=True` devuelve además un tensor `(lote, pasos_generados, posiciones_entrada)`. Los pesos de pasos posteriores al final de una frase se ponen a cero para la visualización.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, source_lengths, target, teacher_forcing_ratio=0.5):
        encoder_outputs, hidden, cell = self.encoder(source, source_lengths)
        source_mask = source != PAD
        token_ids = target[:, 0]  # <sos>
        outputs = []

        for step in range(1, target.size(1)):
            logits, hidden, cell, weights = self.decoder(
                token_ids, hidden, cell, encoder_outputs, source_mask)
            outputs.append(logits)
            if random.random() < teacher_forcing_ratio:
                token_ids = target[:, step]
            else:
                token_ids = logits.argmax(dim=-1)

        # (lote, longitud_objetivo_sin_sos, vocabulario_español)
        return torch.stack(outputs, dim=1)

    @torch.no_grad()
    def generate(self, source, source_lengths, max_new_tokens=50, return_attention=False):
        # El llamador debe activar model.eval(). No se utiliza la traducción real.
        encoder_outputs, hidden, cell = self.encoder(source, source_lengths)
        source_mask = source != PAD
        batch_size = source.size(0)
        token_ids = torch.full((batch_size,), SOS, dtype=torch.long, device=source.device)
        finished = torch.zeros(batch_size, dtype=torch.bool, device=source.device)
        predictions = []
        attention_steps = []

        for _ in range(max_new_tokens):
            logits, hidden, cell, weights = self.decoder(
                token_ids, hidden, cell, encoder_outputs, source_mask)
            # <pad> y <sos> no son palabras que deban generarse.
            logits[:, PAD] = -torch.inf
            logits[:, SOS] = -torch.inf
            next_ids = logits.argmax(dim=-1)
            next_ids = torch.where(finished, PAD, next_ids)
            predictions.append(next_ids)
            # Las filas posteriores al <eos> se rellenan con ceros en el mapa.
            visible_weights = weights.masked_fill(finished.unsqueeze(1), 0.0)
            attention_steps.append(visible_weights)
            finished = finished | (next_ids == EOS)
            token_ids = next_ids
            if finished.all().item():
                break

        predictions = torch.stack(predictions, dim=1)
        if return_attention:
            return predictions, torch.stack(attention_steps, dim=1)
        return predictions


EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2

if baseline is not None:
    EMBEDDING_DIM = baseline["config"]["embedding_dim"]
    HIDDEN_DIM = baseline["config"]["hidden_dim"]
    NUM_LAYERS = baseline["config"]["num_layers"]

# Ambos modelos se entrenan desde cero; no cargamos los pesos sin atención.
random.seed(SEED)
torch.manual_seed(SEED)
encoder = Encoder(len(src_vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
decoder = Decoder(len(tgt_vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
model = Seq2Seq(encoder, decoder).to(DEVICE)
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters()):,}")

## 6. Entrenamiento y selección mediante validación

En entrenamiento usamos teacher forcing parcial. En validación calculamos la pérdida con teacher forcing completo: mide la predicción del siguiente token dado el prefijo correcto y permite seleccionar el modelo sin aleatoriedad en la realimentación. **No mide por sí sola la calidad de la generación libre**, que evaluaremos después.

Sumamos la pérdida de los tokens válidos y la dividimos entre su número. Así, la pérdida media no depende directamente de cuánto relleno haya en un lote. Recortamos la norma de los gradientes para limitar actualizaciones inestables.

Una época recorre todo el conjunto de entrenamiento. El tiempo depende del equipo y del tamaño elegido; empieza con menos pares para comprobar el flujo.

In [ ]:
LEARNING_RATE = 0.001
NUM_EPOCHS = 15
TEACHER_FORCING_RATIO = 0.5

if baseline is not None:
    LEARNING_RATE = baseline["config"]["learning_rate"]
    NUM_EPOCHS = baseline["config"]["num_epochs"]
    TEACHER_FORCING_RATIO = baseline["config"]["teacher_forcing_ratio"]

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_tokens = 0
    ratio = TEACHER_FORCING_RATIO if training else 1.0

    with torch.set_grad_enabled(training):
        for source, lengths, target in loader:
            source, target = source.to(DEVICE), target.to(DEVICE)
            if training:
                optimizer.zero_grad()

            logits = model(source, lengths, target, teacher_forcing_ratio=ratio)
            expected = target[:, 1:]  # No intentamos predecir <sos>.
            loss_sum = criterion(logits.reshape(-1, logits.size(-1)), expected.reshape(-1))
            n_tokens = (expected != PAD).sum().item()
            loss = loss_sum / n_tokens

            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss_sum.item()
            total_tokens += n_tokens

    return total_loss / total_tokens


history = {"train": [], "valid": []}
best_valid_loss = math.inf
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.perf_counter()
    train_loss = run_epoch(model, train_loader, optimizer)
    valid_loss = run_epoch(model, valid_loader)
    history["train"].append(train_loss)
    history["valid"].append(valid_loss)
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = {name: tensor.detach().cpu().clone()
                      for name, tensor in model.state_dict().items()}
    print(f"Época {epoch:02d} | entrenamiento {train_loss:.3f} | "
          f"validación {valid_loss:.3f} | {time.perf_counter() - start:.1f} s")

model.load_state_dict(best_state)
model.eval()

plt.plot(range(1, NUM_EPOCHS + 1), history["train"], label="Entrenamiento (TF parcial)")
plt.plot(range(1, NUM_EPOCHS + 1), history["valid"], label="Validación (TF completo)")
plt.xlabel("Época")
plt.ylabel("Entropía cruzada media por token")
plt.legend()
plt.show()

## 7. Traducción sin teacher forcing

Reutilizamos la misma función de traducción y los mismos ejemplos fijos. La búsqueda sigue siendo voraz, con el mismo límite de longitud. Toda la diferencia está dentro del modelo: el decoder consulta ahora la memoria del encoder en cada paso.

In [ ]:
@torch.no_grad()
def translate(sentence, model, max_new_tokens=50):
    model.eval()
    tokens = tokenize(sentence)
    source = torch.tensor([src_vocab.encode(tokens) + [EOS]], device=DEVICE)
    lengths = torch.tensor([source.size(1)])
    prediction = model.generate(source, lengths, max_new_tokens)
    words = tgt_vocab.decode(prediction[0].tolist())
    unknown = [token for token in tokens if token not in src_vocab.stoi]
    return " ".join(words), unknown


# Ejemplos fijos de prueba: no seleccionamos solo los que salen bien.
for en, es in test_pairs[:8]:
    prediction, unknown = translate(" ".join(en), model)
    print("EN:", " ".join(en))
    print("Referencia:", " ".join(es))
    print("Con atención:", prediction)
    print("Palabras desconocidas:", unknown)
    print()

## 8. Evaluación de generación libre

Reutilizamos la evaluación de toda la prueba, las referencias originales y las métricas opcionales del notebook anterior. La coincidencia exacta es muy estricta y no debe usarse como única medida de calidad. Después compararemos las mismas frases y los resultados por longitud.

In [ ]:
@torch.no_grad()
def evaluate_generation(model, loader, pairs):
    model.eval()
    records = []
    offset = 0
    for source, lengths, target in loader:
        predicted = model.generate(source.to(DEVICE), lengths, max_new_tokens=MAX_WORDS + 10)
        for ids in predicted.cpu().tolist():
            en, es = pairs[offset]
            hypothesis = " ".join(tgt_vocab.decode(ids))
            reference = " ".join(es)  # Referencia original, sin convertir palabras a <unk>.
            records.append({"source": " ".join(en), "reference": reference,
                            "prediction": hypothesis, "source_length": len(en)})
            offset += 1
    return records


test_results = evaluate_generation(model, test_loader, test_pairs)
for lower, upper in [(1, 10), (11, 20), (21, MAX_WORDS)]:
    group = [r for r in test_results if lower <= r["source_length"] <= upper]
    if group:
        exact = sum(r["prediction"] == r["reference"] for r in group) / len(group)
        print(f"Longitud {lower}–{upper}: {len(group)} frases; coincidencia exacta {exact:.1%}")

try:
    from sacrebleu.metrics import BLEU, CHRF
except ImportError:
    print("Métricas opcionales: instala sacrebleu para calcular BLEU y chrF.")
else:
    hypotheses = [r["prediction"] for r in test_results]
    references = [[r["reference"] for r in test_results]]
    for metric in (BLEU(tokenize="none"), CHRF()):
        print(metric.corpus_score(hypotheses, references))
        print("Configuración:", metric.get_signature())

## 9. Guardar el experimento con atención

Guardamos los mismos elementos que en el notebook anterior en un directorio diferente. No sobrescribimos sus resultados ni sus pesos. El checkpoint representa la arquitectura con atención y no es intercambiable con el checkpoint sin atención.

In [ ]:
import json

ARTIFACT_DIR = Path("artifacts/nb06_bis")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "architecture": "lstm_bahdanau", "seed": SEED, "embedding_dim": EMBEDDING_DIM, "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS, "max_words": MAX_WORDS, "max_pairs": MAX_PAIRS,
    "min_frequency": MIN_FREQUENCY, "max_vocab_size": MAX_VOCAB_SIZE,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS, "teacher_forcing_ratio": TEACHER_FORCING_RATIO,
    "source_file": str(SOURCE_FILE), "target_file": str(TARGET_FILE),
    "torch_version": str(torch.__version__),
}
torch.save({"model_state": best_state, "config": config,
            "src_tokens": src_vocab.itos, "tgt_tokens": tgt_vocab.itos,
            "best_valid_loss": best_valid_loss}, ARTIFACT_DIR / "model.pt")

experiment = {"config": config, "src_tokens": src_vocab.itos, "tgt_tokens": tgt_vocab.itos,
              "train_pairs": train_pairs, "valid_pairs": valid_pairs, "test_pairs": test_pairs,
              "history": history, "test_results": test_results}
(ARTIFACT_DIR / "experiment.json").write_text(
    json.dumps(experiment, ensure_ascii=False, indent=2), encoding="utf-8")
print("Resultados guardados en", ARTIFACT_DIR.resolve())

## 10. Comparación con el modelo sin atención

Solo comparamos si están disponibles los resultados guardados del otro notebook. Comprobamos vocabularios, referencias, entradas y configuración de entrenamiento. Se reutilizan tanto los ejemplos buenos como los malos: no seleccionamos frases según qué modelo gane.

Si SacreBLEU está instalado, mostramos BLEU y chrF globales y por longitud. Una sola ejecución no garantiza una mejora: también influyen el tamaño de la muestra, la convergencia y la semilla. Para evaluar la variabilidad, repite ambos entrenamientos con varias semillas.

In [ ]:
if baseline is None:
    print("Ejecuta nb05_bis.ipynb y guarda sus resultados para realizar la comparación.")
else:
    assert src_vocab.itos == baseline["src_tokens"]
    assert tgt_vocab.itos == baseline["tgt_tokens"]
    old_results = baseline["test_results"]
    assert len(old_results) == len(test_results)
    assert [(r["source"], r["reference"]) for r in old_results] == [
        (r["source"], r["reference"]) for r in test_results]
    for key in ("seed", "embedding_dim", "hidden_dim", "num_layers", "batch_size",
                "learning_rate", "num_epochs", "teacher_forcing_ratio"):
        assert config[key] == baseline["config"][key], f"Configuración distinta: {key}"

    for old, new in zip(old_results[:8], test_results[:8]):
        print("EN:", new["source"])
        print("Referencia:", new["reference"])
        print("Sin atención:", old["prediction"])
        print("Con atención:", new["prediction"])
        print()

    try:
        from sacrebleu.metrics import BLEU, CHRF
    except ImportError:
        print("Instala sacrebleu para comparar también BLEU y chrF.")
    else:
        for lower, upper in [(1, MAX_WORDS), (1, 10), (11, 20), (21, MAX_WORDS)]:
            indices = [i for i, r in enumerate(test_results)
                       if lower <= r["source_length"] <= upper]
            if not indices:
                continue
            refs = [[test_results[i]["reference"] for i in indices]]
            print(f"Longitud {lower}–{upper}: {len(indices)} frases")
            for label, results in [("Sin atención", old_results), ("Con atención", test_results)]:
                hypotheses = [results[i]["prediction"] for i in indices]
                bleu = BLEU(tokenize="none").corpus_score(hypotheses, refs).score
                chrf = CHRF().corpus_score(hypotheses, refs).score
                print(f"  {label}: BLEU={bleu:.2f}; chrF={chrf:.2f}")

## 11. Visualizar la atención

Cada fila corresponde a un token generado y cada columna a una posición de la entrada, incluido `<eos>`. El color representa el peso que recibió esa posición al predecir ese token. Se usa la generación libre, no la referencia española.

El mapa ayuda a inspeccionar el mecanismo, pero no demuestra por sí solo una alineación lingüística correcta. Una atención difusa o una traducción incorrecta también son resultados que conviene analizar.

In [ ]:
@torch.no_grad()
def plot_attention(sentence, model, max_new_tokens=50):
    model.eval()
    source_tokens = tokenize(sentence) + ["<eos>"]
    source_ids = src_vocab.encode(source_tokens)
    source = torch.tensor([source_ids], dtype=torch.long, device=DEVICE)
    lengths = torch.tensor([len(source_ids)])
    predictions, attention = model.generate(
        source, lengths, max_new_tokens=max_new_tokens, return_attention=True)

    generated_ids = predictions[0].tolist()
    if EOS in generated_ids:
        generated_ids = generated_ids[:generated_ids.index(EOS) + 1]
    generated_tokens = [tgt_vocab.itos[index] for index in generated_ids]
    weights = attention[0, :len(generated_ids), :len(source_tokens)].cpu()
    # Mostramos <unk> cuando esa fue realmente la entrada al modelo.
    source_labels = [src_vocab.itos[index] for index in source_ids]

    fig, ax = plt.subplots(figsize=(max(7, len(source_tokens) * 0.4),
                                    max(3, len(generated_tokens) * 0.35)))
    image = ax.imshow(weights.numpy(), aspect="auto", vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(source_labels)), labels=source_labels, rotation=90)
    ax.set_yticks(range(len(generated_tokens)), labels=generated_tokens)
    ax.set_xlabel("Tokens de entrada (inglés)")
    ax.set_ylabel("Tokens generados (español)")
    ax.set_title("Atención de Bahdanau durante la generación")
    fig.colorbar(image, ax=ax, label="Peso de atención")
    fig.tight_layout()
    plt.show()
    return weights


attention_weights = plot_attention(" ".join(test_pairs[0][0]), model)